In [1]:
# Get raw data
with open('input/16.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
import numpy as np

In [ ]:
# Part 1
# Get maze layout
layout = np.array([list(i) for i in rawinput.split('\n')])

# Identify corridor junctions (+start/end point) and assign each a unique ID
junctions = np.where(layout=='#', 0,
                     np.sum(np.stack(((z:=np.pad(layout,1))[:-2,1:-1]!='#',
                                      z[2:,1:-1]!='#',
                                      z[1:-1,:-2]!='#',
                                      z[1:-1,2:]!='#')),
                            axis=0)) > 2
junctions = np.where(junctions, 
                     np.cumsum(junctions.reshape(-1)).reshape(junctions.shape), 
                     -1)
junctions[-2,1] = 0
junctions[1,-2] = np.max(junctions)+1

# For each "adjacent" pair of junction/heading states, calculate cost to travel between them
# (Also capture list of individual cell coords, for part 2)
mv_opt = np.array([[-1,0],[1,0],[0,-1],[0,1]])
jnc_ix = np.arange(layout.size)[(jf:=junctions.reshape(-1))!=-1]
jnc_start = np.repeat((np.column_stack((jnc_ix//(z:=layout.shape[1]),jnc_ix%z))),4,axis=0)
jnc_mv = np.tile(np.arange(4),jnc_ix.shape[0])
jnc_start,jnc_mv = [i[f]
                    for f in [((layout[tuple(jnc_start.T)]=='S') & (jnc_mv==3))
                              | (layout[tuple((jnc_start-mv_opt[jnc_mv]).T)]!='#')]
                    for i in [jnc_start,jnc_mv]]

pos = np.expand_dims(jnc_start,1)
mv = np.copy(jnc_mv)
cost = np.zeros_like(mv)
path_cells = {}
path_cost = {}
while jnc_start.shape[0]:
    nr = pos.shape[0]
    jnc_start, jnc_mv, pos, mv, cost = [np.repeat(i,4,axis=0) 
                                        for i in [jnc_start, jnc_mv, pos, mv, cost]]

    mvflags = np.max(np.abs(mv_opt[mv]+mv_opt[(mv:=np.tile(np.arange(4),nr))]), axis=1)
    pos = np.append(pos, np.expand_dims(pos[:,-1]+mv_opt[mv],1), axis=1)
    cost += (2001-1000*mvflags)

    jnc_start, jnc_mv, pos, mv, cost = [i[f]
                                        for f in [(layout[tuple(pos[:,-1].T)] != '#') & (mvflags > 0)]
                                        for i in [jnc_start, jnc_mv, pos, mv, cost]]

    jnc_start_num = junctions[tuple(jnc_start.T)]
    jnc_end_num = junctions[tuple(pos[:,-1].T)]
    f = jnc_end_num!=-1
    for ix in np.arange(f.shape[0])[f]:
        sj,sd,ej,ed = [i[ix].item() for i in [jnc_start_num, jnc_mv, jnc_end_num, mv]]
        if (sj,sd) not in path_cells:
            path_cells[(sj,sd)] = {}
            path_cost[(sj,sd)] = {}
        path_cells[(sj,sd)][(ej,ed)] = pos[ix]
        path_cost[(sj,sd)][(ej,ed)] = cost[ix].item()
        
    jnc_start, jnc_mv, pos, mv, cost = [i[~f] for i in [jnc_start, jnc_mv, pos, mv, cost]]
    
# Iteratively search possible routes, eliminating all but the lowest cost route to
# each state in each step.  Fetch lowest cost for exit cell.
state_poss = [[((0,3),0)]]
state_mincost = dict(state_poss[0])
best_routes = []
while state_poss:
    state_poss = [p+[(nextpos,cost+addcost)]
                  for p in state_poss
                  for pos,cost in [p[-1]]
                  for nextpos,addcost in path_cost.get(pos,{}).items()]
    for i in range(len(state_poss)-1,-1,-1):
        if state_poss[i][-1][0] in state_mincost:
            if (z:=state_poss[i][-1][1]-state_mincost[state_poss[i][-1][0]])>1:
                state_poss.pop(i)
            elif z==0:
                if state_poss[i][-1][0][0] == np.max(junctions):
                    best_routes += [state_poss[i]]
            else:
                state_mincost[state_poss[i][-1][0]] = state_poss[i][-1][1]
                if state_poss[i][-1][0][0] == np.max(junctions):
                    best_routes = [state_poss[i]]
        else:
            state_mincost[state_poss[i][-1][0]] = state_poss[i][-1][1]
            if state_poss[i][-1][0][0] == np.max(junctions):
                best_routes = [state_poss[i]]
                
min([v for k,v in state_mincost.items() if k[0]==np.max(junctions)])

130536

In [4]:
# Part 2
len({tuple(l)
     for i in best_routes 
     for j,(k,_) in enumerate(i[1:]) 
     for l in path_cells[i[j][0]][k].tolist()})

1024